# NB3 — RAG Copilot Integration

Builds the ARIA knowledge base and wires retrieval into a diagnostic report generator.

**Your RAG project plugs in at 4 `[SWAP_RAG]` markers:**

| Marker | What goes there |
|---|---|
| `TFIDFRetriever` | Your vector store + embedder |
| `ingest_pdf_datasheet()` | Your PDF ingestion pipeline |
| `analyse_relay_image()` | Your image understanding model |
| Report generation | Your LLM + retrieved context |

**Outputs:** `aria_kb/` — 11 KB documents + 3 sample diagnostic reports

**ARIA — Relay Intelligence · APOGEE'26 · BITS Pilani · Luminous Power Technologies**

## Installation

In [ ]:
!pip install numpy pandas

## Setup

In [ ]:
"""
ARIA — Adaptive Relay Intelligence & Anomaly System
Notebook 3: RAG Copilot Integration

This notebook does three things:
  1. Builds the knowledge base (KB) from relay datasheets, service manuals,
     failure case studies, and MODBUS metadata from Notebook 1.
  2. Implements a retrieval engine (TF-IDF + cosine similarity offline;
     swap to your own RAG embedder in production using the [SWAP_RAG] markers).
  3. Wires the retrieval into a diagnostic report generator — the output
     that gets pushed to the mobile app when an alert fires.

Your RAG project plugs in at [SWAP_RAG] markers:
  - PDF ingestion   → replace _load_pdf_stub() with your PDF parser
  - Image understanding → replace _analyse_image_stub() with your vision model
  - Table/chart parsing → replace _parse_table_stub() with your table extractor
  - Embedding/retrieval → replace TFIDFRetriever with your vector store

Reads:  aria_data/metadata.json  (MODBUS map from Notebook 1)
        aria_models/model_registry.json
Writes: aria_kb/       (knowledge base documents)
        aria_kb/kb_index.json
"""

import os, json, re, math, joblib
from datetime import datetime
from collections import defaultdict
import numpy as np
import pandas as pd

KB_DIR    = "aria_kb"
MODEL_DIR = "aria_models"
DATA_DIR  = "aria_data"
os.makedirs(KB_DIR, exist_ok=True)

## Section

In [ ]:
# 1. KNOWLEDGE BASE DOCUMENTS
# In production these come from:
#   - Real relay datasheets (PDF ingestion via your RAG)
#   - Luminous service manuals (PDF ingestion)
#   - Field failure case studies (structured JSON)
#   - MODBUS register map (from Notebook 1 metadata.json)
# Here we write realistic synthetic KB documents.

## Section

In [ ]:
KB_DOCUMENTS = [

    # ── Relay datasheet knowledge ──────────────────────────────────────────
    {
        "id": "ds_001",
        "type": "datasheet",
        "title": "Omron G2R Series — Relay Specification",
        "content": """
The Omron G2R series relay is rated for 100,000 switching operations at rated load
(10A, 250VAC). At reduced loads, switching life increases significantly: at 5A the
rated life is 300,000 operations. Life decreases exponentially with current above
rated load due to increased contact arc energy. Coil operating voltage is 12VDC with
allowable voltage range 10–14V. Contact resistance when new is 50 milliohms maximum.
Contact resistance above 200 milliohms indicates moderate wear; above 400 milliohms
indicates severe wear and imminent failure. Bounce time when new is 1–3 milliseconds.
Bounce time exceeding 6 milliseconds indicates mechanical spring fatigue and is a
strong predictor of imminent failure. Operating temperature range is -40°C to +70°C.
For every 10°C rise above 25°C, relay life degrades by approximately 50% due to
accelerated coil insulation breakdown and contact oxidation (Arrhenius relationship).
Recommended replacement interval: proactive replacement at 80,000 cycles or when
health index drops below 30%, whichever comes first. Contact pitting visible as
small craters on contact surface after 40,000+ operations under high load.
""",
        "tags": ["relay", "datasheet", "life", "contact", "bounce", "temperature", "omron"]
    },

    {
        "id": "ds_002",
        "type": "datasheet",
        "title": "Schneider LC1-E Contactor — Technical Data",
        "content": """
The Schneider LC1-E series contactor is rated for 1,000,000 mechanical operations
and 100,000 electrical operations at AC3 duty (full load motor switching). The
primary failure modes are: (1) contact erosion from arc energy, causing increased
contact drop voltage and resistance; (2) coil burnout from sustained overvoltage
or high ambient temperature; (3) spring fatigue causing slow response and increased
bounce duration. Typical contact bounce for a new contactor is 5–10ms. Bounce
exceeding 20ms is cause for replacement. Contact resistance threshold for replacement
is 500 milliohms. When switching AC motor loads, inrush current is typically 6–8
times rated current for 100–200ms, causing disproportionate arc damage. Each motor
start therefore counts as approximately 36–64 equivalent standard switching cycles
for wear estimation purposes. Environmental factors: dust accumulation on contacts
increases arc intensity and accelerates erosion. Humidity above 85% RH causes
contact oxidation and increased resistance even without switching.
""",
        "tags": ["contactor", "datasheet", "motor", "inrush", "arc", "schneider", "erosion"]
    },

    # ── Failure case studies ──────────────────────────────────────────────
    {
        "id": "cs_001",
        "type": "case_study",
        "title": "Case Study: AC Compressor Inrush — Premature Relay Failure",
        "content": """
Site: Residential installation, Jaipur, Rajasthan. 1.5 ton split AC connected to
2kVA Luminous inverter. Relay failed after 8 months (expected life: 3 years).
Root cause analysis: AC compressor draws 12–15A inrush on every start. Customer
reported 6–8 AC starts per day (morning and evening cooling cycles). Actual switching
current at relay: measured 13.5A peak. At this current, arc energy per switch is
approximately 18× normal (I² scaling). Effective wear per switch: 18 equivalent
cycles. Total equivalent cycles in 8 months: 8 starts/day × 245 days × 18 = 35,280
cycles, close to rated 100,000 but accelerated further by 44°C ambient in summer.
Arrhenius factor at 44°C: 2.8×. Effective cycles: 35,280 × 2.8 = 98,784. Failure
consistent with rated life reached. Recommendation: for customers with 1+ AC units,
trigger yellow alert at 40,000 equivalent cycles and orange at 70,000. Pre-schedule
relay replacement before monsoon season (high humidity accelerates final failure).
Contact surface showed classic crater pitting pattern. Visual: severe_erosion stage.
""",
        "tags": ["case_study", "AC", "inrush", "failure", "jaipur", "residential", "compressor"]
    },

    {
        "id": "cs_002",
        "type": "case_study",
        "title": "Case Study: Frequent Power Cuts — Contact Welding Failure",
        "content": """
Site: Small commercial shop, Uttar Pradesh. 15–20 power interruptions per day during
summer. Inverter: 3kVA with mixed resistive and motor load. Relay failure mode:
contact welding (contacts fused together, relay stuck in closed position). This is
different from erosion failure — welding occurs when switching current is extremely
high at moment of contact opening. Analysis: shop had inductive loads (fans, small
motors) that create voltage spikes when switched. Relay switching at 8–10A with
inductive kickback spikes to 15–18A. At 18 power cuts per day: 18 × 365 = 6,570
operations per year. Welding failure occurred at month 14 (9,590 operations), far
below rated 100,000. Indicates severe inductive load damage per switch. Symptoms
before failure: clicking sound without power transfer (contacts bouncing on weld
spots), intermittent supply, 200ms+ transfer delay. Algorithm note: flag contact
welding risk separately from erosion — monitor for: high arc energy + low switch count
+ sudden zero bounce time (welded contacts show no bounce). Recommend surge arrestor
installation when inductive load detected.
""",
        "tags": ["case_study", "welding", "inductive", "power_cuts", "commercial", "UP", "symptom"]
    },

    {
        "id": "cs_003",
        "type": "case_study",
        "title": "Case Study: Hot Environment — Coil Burnout Pattern",
        "content": """
Site: Industrial pump house, Gujarat. Ambient temperature 50–55°C. Relay coil
burnout after 6 months (expected: 2+ years). Switching frequency was low (4 per day)
but continuous coil energisation at high temperature caused insulation breakdown.
Coil resistance increased from 240 ohms (new) to 310 ohms before failure, indicating
insulation degradation. Key diagnostic: coil current draw increased by 30% in final
weeks. Temperature sensor reading above 48°C sustained for >6 hours daily is a
strong predictor of accelerated coil failure independent of switching count.
Recommendation: in high-temperature environments, issue yellow alert when:
temperature > 45°C sustained AND health index < 60%. Issue orange alert when
temperature > 50°C sustained AND health index < 80%. Pre-failure symptoms:
intermittent relay click without energisation, increased coil hum, warm smell from
relay housing. Visual degradation: discoloration of relay housing, yellowing of coil
housing visible in maintenance inspection photograph.
""",
        "tags": ["case_study", "temperature", "coil", "burnout", "industrial", "gujarat", "heat"]
    },

    {
        "id": "cs_004",
        "type": "case_study",
        "title": "Case Study: Dust and Humidity — Contact Oxidation",
        "content": """
Site: Coastal residential, Kerala. High humidity (85–95% RH). Inverter in outdoor
utility area with dust exposure. Relay contact resistance increased from 50mΩ to
380mΩ over 18 months without high switching frequency (3 cuts per day). Root cause:
contact oxidation layer formed due to humidity, then partially burned off at each
switch, depositing carbon residue. Cycle repeats, layering resistance. Symptoms:
flickering supply during transfer, warm contacts, relay buzzing during hold.
Diagnostic pattern: contact resistance rising steadily (>5mΩ per month increase)
despite low equivalent switching cycles. This distinguishes oxidation from arc
erosion — erosion shows high arc energy, oxidation shows high resistance with low
arc energy. Algorithm trigger: flag oxidation risk when contact_resistance_mOhm
increase rate > 5 per month AND arc_energy_sum_30d < threshold. Recommend sealed
relay housing inspection and contact cleaning at 200mΩ threshold. Visual:
greenish-grey discoloration on contacts versus crater pitting of arc erosion.
""",
        "tags": ["case_study", "humidity", "oxidation", "coastal", "kerala", "dust", "resistance"]
    },

    # ── Service procedures ────────────────────────────────────────────────
    {
        "id": "sp_001",
        "type": "service_procedure",
        "title": "Relay Replacement Procedure — Luminous Inverter",
        "content": """
Tools required: Phillips screwdriver, needle-nose pliers, multimeter, anti-static
wrist strap. Replacement relay: ensure exact coil voltage match (12VDC for most
Luminous models). Estimated time: 20–30 minutes for trained technician.

Step 1: Ensure inverter is OFF and disconnected from mains and battery.
Step 2: Remove inverter cover (4–6 screws on rear panel).
Step 3: Locate relay on PCB — typically near AC input section, marked RLY1 or K1.
Step 4: Measure contact resistance with multimeter across COM and NO terminals.
  If < 100mΩ: relay may still be serviceable; verify bounce time.
  If 100–300mΩ: schedule replacement within 30 days.
  If > 300mΩ: immediate replacement recommended.
Step 5: Desolder existing relay (4 pins + 2 coil pins). Use desoldering pump.
Step 6: Install new relay. Verify orientation matches PCB silkscreen.
Step 7: Solder all pins. Check for cold joints.
Step 8: Power test: verify transfer time < 20ms, no audible bounce, stable output.
Step 9: Log replacement in service record with date and cycle count at replacement.

Safety note: Do NOT touch capacitors on PCB immediately after power off — allow
5 minutes for discharge. Capacitors may retain charge up to 400V.
""",
        "tags": ["service", "replacement", "procedure", "PCB", "technician", "solder"]
    },

    {
        "id": "sp_002",
        "type": "service_procedure",
        "title": "Contact Cleaning Procedure (Oxidation Cases)",
        "content": """
When contact resistance is elevated (150–300mΩ) due to oxidation rather than arc
erosion, contact cleaning may extend relay life by 6–12 months.

Tools: Contact cleaner spray (non-corrosive, e.g. CRC QD Electronic Cleaner),
clean lint-free cloth, compressed air.

Step 1: Power off inverter, wait 5 minutes for capacitor discharge.
Step 2: Access relay as per replacement procedure steps 1–3.
Step 3: If relay is socketed: remove from socket and spray contacts with cleaner.
  If relay is soldered: spray contact area carefully avoiding PCB components.
Step 4: Allow to dry completely (minimum 10 minutes).
Step 5: Test contact resistance. Should return to < 100mΩ if oxidation is cause.
Step 6: If resistance remains > 200mΩ after cleaning, proceed to full replacement.
Step 7: Log cleaning date and pre/post resistance values.

Note: Cleaning does NOT address arc erosion (crater pitting). If visual inspection
shows pitting or if switching cycle count is high, replace the relay regardless
of current resistance reading.
""",
        "tags": ["service", "cleaning", "oxidation", "contact", "resistance", "maintenance"]
    },

    # ── MODBUS and system knowledge ───────────────────────────────────────
    {
        "id": "sys_001",
        "type": "system_knowledge",
        "title": "ARIA Health Index Interpretation Guide",
        "content": """
The ARIA Health Index (HI) is a 0–100 score computed from the physics-based wear
model incorporating arc energy accumulation, thermal stress, and contact resistance.

HI 80–100 (GREEN): Relay is healthy. No action required.
  Typical characteristics: bounce < 3ms, contact resistance < 100mΩ, arc energy
  per switch within normal range for load profile.

HI 60–79 (LIGHT GREEN): Relay showing early wear. Schedule inspection at next
  routine service visit. No urgency.

HI 30–59 (YELLOW): Moderate wear detected. Plan replacement within 60 days.
  Customer notification: "Your inverter's switching component is at moderate health.
  We recommend scheduling a service visit in the next 1–2 months."

HI 10–29 (ORANGE): Significant wear. Schedule replacement within 14 days.
  Risk of intermittent supply or transfer delays if not addressed.
  Customer notification: "Your inverter needs attention. Please schedule service
  within 2 weeks to avoid unexpected power issues."

HI 0–9 (RED): Critical. Replacement urgently required.
  Failure may be imminent. Symptoms may include clicking without transfer,
  flickering supply, or complete failure to switch.
  Customer notification: "Urgent: Your inverter's relay is at critical health.
  Please contact service immediately to avoid power outage."

RUL (Remaining Useful Life) is provided with 90% confidence interval.
Example: "RUL: 47 days [38–61 days]" means with 90% probability the relay
will remain functional for 38 to 61 more days under current operating conditions.
""",
        "tags": ["health_index", "RUL", "interpretation", "alert", "customer", "notification"]
    },

    {
        "id": "sys_002",
        "type": "system_knowledge",
        "title": "Failure Mode Classification — Relay Diagnostics",
        "content": """
ARIA classifies relay degradation into four primary failure modes:

1. ARC_EROSION (most common, ~60% of failures)
   Indicators: high arc_energy per switch, high equivalent cycles, crater pitting visible.
   Cause: high switching current, especially AC motor inrush.
   Pattern: gradual HI decline correlated with switching events.
   Action: schedule proactive replacement based on RUL estimate.

2. THERMAL_DEGRADATION (~20% of failures)
   Indicators: temperature > 45°C sustained, coil resistance increasing.
   Cause: hot environment, poor ventilation, continuous energisation.
   Pattern: HI decline faster in summer months, slower in winter.
   Action: also address root cause (ventilation, shade, derating).

3. CONTACT_OXIDATION (~15% of failures)
   Indicators: rising contact resistance with low arc energy, coastal/humid location.
   Cause: humidity, dust, salt air.
   Pattern: resistance rises steadily, independent of switching frequency.
   Action: contact cleaning first; replacement if cleaning insufficient.

4. CONTACT_WELDING (rare, ~5%, but sudden)
   Indicators: very high arc energy spike, inductive load, sudden zero bounce time.
   Cause: inductive load switching, voltage spikes.
   Pattern: sudden failure rather than gradual; Isolation Forest flags anomaly.
   Action: immediate replacement; install surge protection.
""",
        "tags": ["failure_mode", "classification", "arc", "thermal", "oxidation", "welding"]
    },
]

## Section

In [ ]:
# 2. KNOWLEDGE BASE BUILDER

## Section

In [ ]:
def build_knowledge_base():
    """Save KB documents to disk and build TF-IDF index."""
    print("Building knowledge base...")

    # Save individual documents
    for doc in KB_DOCUMENTS:
        path = os.path.join(KB_DIR, f"{doc['id']}.json")
        with open(path, "w") as f:
            json.dump(doc, f, indent=2)

    # Also ingest MODBUS metadata from Notebook 1
    meta_path = os.path.join(DATA_DIR, "metadata.json")
    if os.path.exists(meta_path):
        with open(meta_path) as f:
            meta = json.load(f)
        modbus_doc = {
            "id": "sys_003",
            "type": "system_knowledge",
            "title": "Inverter MODBUS Register Map",
            "content": "Available MODBUS registers:\n" + "\n".join(
                f"Register {addr}: {name}" for addr, name in meta.get("modbus_map", {}).items()
            ),
            "tags": ["modbus", "register", "inverter", "protocol"]
        }
        with open(os.path.join(KB_DIR, "sys_003.json"), "w") as f:
            json.dump(modbus_doc, f, indent=2)
        KB_DOCUMENTS.append(modbus_doc)
        print(f"  Ingested MODBUS map → sys_003.json")

    print(f"  {len(KB_DOCUMENTS)} documents in knowledge base")
    return KB_DOCUMENTS

## Section

In [ ]:
# 3. TF-IDF RETRIEVER
# [SWAP_RAG] Replace this entire class with your RAG embedder:
#   from your_rag import embed, retrieve
#   class YourRAGRetriever:
#       def __init__(self, docs): self.index = embed(docs)
#       def retrieve(self, query, k): return retrieve(self.index, query, k)

## Section

In [ ]:
class TFIDFRetriever:
    """
    Offline TF-IDF retriever.
    [SWAP_RAG] In production replace with your multimodal RAG embedder.
    Your RAG handles: PDF ingestion, table/chart parsing, image understanding.
    Same interface: __init__(docs) + retrieve(query, k) → list of docs.
    """
    def __init__(self, documents: list):
        self.docs = documents
        self.vocab, self.idf, self.tf_idf_matrix = self._build_index()

    def _tokenize(self, text: str) -> list:
        return re.findall(r'\b[a-z]{3,}\b', text.lower())

    def _build_index(self):
        N = len(self.docs)
        # Build vocabulary
        all_tokens = set()
        doc_tokens = []
        for doc in self.docs:
            tokens = self._tokenize(doc["title"] + " " + doc["content"] + " " + " ".join(doc["tags"]))
            doc_tokens.append(tokens)
            all_tokens.update(tokens)
        vocab = {w: i for i, w in enumerate(sorted(all_tokens))}
        V = len(vocab)

        # TF matrix
        tf = np.zeros((N, V), dtype=np.float32)
        for i, tokens in enumerate(doc_tokens):
            counts = defaultdict(int)
            for t in tokens: counts[t] += 1
            total = max(len(tokens), 1)
            for t, c in counts.items():
                if t in vocab: tf[i, vocab[t]] = c / total

        # IDF
        df = (tf > 0).sum(axis=0) + 1
        idf = np.log((N + 1) / df).astype(np.float32)

        # TF-IDF
        tfidf = tf * idf
        # L2-normalise
        norms = np.linalg.norm(tfidf, axis=1, keepdims=True) + 1e-8
        tfidf = tfidf / norms
        return vocab, idf, tfidf

    def retrieve(self, query: str, k: int = 3) -> list:
        tokens = self._tokenize(query)
        q_vec  = np.zeros(len(self.vocab), dtype=np.float32)
        for t in tokens:
            if t in self.vocab: q_vec[self.vocab[t]] += 1
        q_vec *= self.idf
        norm = np.linalg.norm(q_vec) + 1e-8
        q_vec /= norm
        scores = self.tf_idf_matrix @ q_vec
        top_k  = np.argsort(scores)[::-1][:k]
        return [{"doc": self.docs[i], "score": float(scores[i])} for i in top_k if scores[i] > 0]

## Section

In [ ]:
# 4. MULTIMODAL STUBS
# [SWAP_RAG] Replace stubs with your actual RAG capabilities

## Section

In [ ]:
def analyse_relay_image(image_path: str, retriever: TFIDFRetriever) -> dict:
    """
    [SWAP_RAG] Replace with your image-understanding module.
    Your RAG: pass image to vision model → extract visual degradation features
    → retrieve matching case study → return structured diagnosis.

    Stub behaviour: returns a plausible result based on filename convention.
    Expected image naming: relay_<device_id>_<visual_label>.jpg
    """
    # Extract visual label from filename if present (for demo)
    visual_label = "unknown"
    for label in ["new", "mild_pitting", "moderate_erosion", "severe_erosion", "pre_failure"]:
        if label in str(image_path):
            visual_label = label
            break

    # [SWAP_RAG] Your actual implementation:
    # image_embedding = your_vision_model.encode(image_path)
    # visual_label = your_classifier.predict(image_embedding)
    # similar_cases = retriever.retrieve(f"relay contact {visual_label}", k=2)

    query = f"relay contact {visual_label} visual inspection"
    similar_cases = retriever.retrieve(query, k=2)

    severity_map = {
        "new":              ("GREEN",  "No visual degradation. Contacts look new.", 95),
        "mild_pitting":     ("YELLOW", "Minor contact pitting visible. Early arc erosion.", 65),
        "moderate_erosion": ("ORANGE", "Moderate crater pitting. Schedule replacement.", 40),
        "severe_erosion":   ("RED",    "Severe erosion. Immediate replacement.", 15),
        "pre_failure":      ("RED",    "Pre-failure state. Replace now.", 5),
        "unknown":          ("UNKNOWN","Could not classify image.", 50),
    }
    colour, description, hi_estimate = severity_map[visual_label]

    return {
        "visual_label":  visual_label,
        "severity":      colour,
        "description":   description,
        "hi_estimate":   hi_estimate,
        "similar_cases": [r["doc"]["title"] for r in similar_cases],
        "note": "[SWAP_RAG] Replace with your_vision_model.encode() + classifier"
    }


def ingest_pdf_datasheet(pdf_path: str) -> dict:
    """
    [SWAP_RAG] Replace with your PDF ingestion module.
    Your RAG: parse PDF → extract text + tables + images → chunk → embed → store.

    Stub: returns metadata about what would be ingested.
    """
    return {
        "status":  "stub",
        "path":    pdf_path,
        "message": "[SWAP_RAG] Replace with your PDF parser. "
                   "Your multimodal RAG handles text extraction, "
                   "table parsing, and embedded image extraction from PDFs.",
        "expected_chunks": ["relay specifications", "contact ratings table",
                            "life expectancy curves", "wiring diagrams"]
    }

## Section

In [ ]:
# 5. DIAGNOSTIC REPORT GENERATOR
# This is what gets pushed to the mobile app when an alert fires

## Section

In [ ]:
def classify_failure_mode(alert: dict) -> str:
    """Determine most likely failure mode from alert context."""
    arc   = alert.get("arc_energy_sum_72h", 0)
    temp  = alert.get("temperature_mean_C", 30)
    resist= alert.get("contact_resistance_mOhm", 50)
    bounce= alert.get("bounce_ms", 2)
    anomaly = alert.get("is_anomaly", False)

    if anomaly and bounce < 0.5:
        return "CONTACT_WELDING"
    if temp > 45 and arc < 1.0:
        return "THERMAL_DEGRADATION"
    if resist > 150 and arc < 0.5:
        return "CONTACT_OXIDATION"
    return "ARC_EROSION"


def build_rag_query(alert: dict, failure_mode: str) -> str:
    """Construct the retrieval query from alert context."""
    hi  = alert.get("health_index", 50)
    rul = alert.get("rul_days", 90)
    scenario_hints = []
    if alert.get("ac_starts_detected", False): scenario_hints.append("AC motor inrush")
    if alert.get("temperature_mean_C", 30) > 45: scenario_hints.append("high temperature environment")
    if alert.get("is_coastal", False): scenario_hints.append("humidity oxidation coastal")
    hint_str = " ".join(scenario_hints) or "standard residential"
    return (f"relay {failure_mode.lower().replace('_',' ')} "
            f"health index {hi:.0f} RUL {rul:.0f} days {hint_str} "
            f"diagnosis recommendation service action")


def generate_diagnostic_report(alert: dict, retriever: TFIDFRetriever,
                                image_path: str = None) -> dict:
    """
    Core RAG pipeline:
    1. Classify failure mode
    2. Build retrieval query
    3. Retrieve relevant KB chunks
    4. [SWAP_RAG] Pass to your LLM + retrieved context for generation
    5. Optionally enrich with image analysis
    6. Return structured report for mobile app
    """
    device_id    = alert.get("device_id", "INV-UNKNOWN")
    hi           = alert.get("health_index", 50)
    rul          = alert.get("rul_days", 90)
    rul_lo       = alert.get("rul_ci_low", rul - 10)
    rul_hi       = alert.get("rul_ci_high", rul + 10)
    failure_mode = classify_failure_mode(alert)

    # Retrieve relevant knowledge
    query       = build_rag_query(alert, failure_mode)
    retrieved   = retriever.retrieve(query, k=3)
    context     = "\n\n".join(r["doc"]["content"].strip() for r in retrieved)

    # Image analysis (if photo provided)
    image_result = None
    if image_path:
        image_result = analyse_relay_image(image_path, retriever)

    # ── Generate report ──────────────────────────────────────────────────
    # [SWAP_RAG] In production:
    # prompt = build_prompt(context, alert, failure_mode)
    # report_text = your_llm.generate(prompt, retrieved_context=context)
    # Here we construct a structured report from templates + retrieved content.

    severity_label = ("CRITICAL" if hi < 10 else "HIGH" if hi < 30 else
                      "MEDIUM"   if hi < 60 else "LOW")
    urgency_days   = (2 if hi < 10 else 14 if hi < 30 else 60 if hi < 60 else 180)

    # Customer-facing plain language (GenAI would generate this in production)
    customer_messages = {
        "ARC_EROSION":        (f"Your inverter's switching relay has been working hard handling "
                               f"heavy loads. Think of it like brake pads — they wear down with use. "
                               f"It's at {hi:.0f}% health with about {rul:.0f} days of life remaining."),
        "THERMAL_DEGRADATION":(f"Your inverter relay is showing heat stress. High ambient temperatures "
                               f"accelerate component wear. Current health: {hi:.0f}%. "
                               f"Estimated remaining life: {rul:.0f} days."),
        "CONTACT_OXIDATION":  (f"Your relay contacts are showing oxidation, likely due to humidity. "
                               f"A contact cleaning may extend life. Health: {hi:.0f}%, "
                               f"estimated life: {rul:.0f} days."),
        "CONTACT_WELDING":    (f"URGENT: Your relay shows signs of contact stress that may cause "
                               f"sudden failure. Immediate inspection recommended."),
    }

    # Engineer-facing technical guidance
    engineer_actions = {
        "ARC_EROSION":        ["Measure contact resistance (expect >150mΩ)",
                               "Check bounce time (expect >4ms)",
                               "Review load profile for motor/AC loads",
                               "Schedule relay replacement", "Log switch count at replacement"],
        "THERMAL_DEGRADATION":["Check ambient temperature at installation site",
                               "Verify inverter ventilation clearances (min 10cm all sides)",
                               "Measure coil resistance (expect >280Ω)",
                               "Replace relay", "Consider relocating inverter to cooler position"],
        "CONTACT_OXIDATION":  ["Inspect contacts visually for greenish discoloration",
                               "Attempt contact cleaning (see SP-002)",
                               "Re-measure resistance post-cleaning",
                               "Replace if resistance > 200mΩ after cleaning",
                               "Seal inverter enclosure if dusty/humid environment"],
        "CONTACT_WELDING":    ["DO NOT attempt manual operation",
                               "Power off inverter immediately",
                               "Replace relay — welded contacts cannot be repaired",
                               "Install surge protection on inductive loads",
                               "Check for voltage spikes on input line"],
    }

    # Find most relevant case study
    relevant_cases = [r["doc"]["title"] for r in retrieved if r["doc"]["type"] == "case_study"]

    report = {
        "report_id":       f"RPT-{device_id}-{datetime.now().strftime('%Y%m%d%H%M%S')}",
        "generated_at":    datetime.now().isoformat(),
        "device_id":       device_id,

        # Alert summary
        "alert": {
            "health_index":  round(hi, 1),
            "severity":      severity_label,
            "rul_days":      round(rul, 1),
            "rul_ci":        [round(rul_lo, 1), round(rul_hi, 1)],
            "failure_mode":  failure_mode,
            "urgency_days":  urgency_days,
        },

        # Customer-facing content (goes to homeowner app)
        "customer": {
            "message":       customer_messages[failure_mode],
            "action":        (f"Schedule service within {urgency_days} days"
                              if urgency_days < 180 else "No action needed now"),
            "plain_rul":     f"About {int(rul)} days remaining ({int(rul_lo)}–{int(rul_hi)} day range)",
        },

        # Engineer-facing content (goes to service engineer app)
        "engineer": {
            "failure_mode":  failure_mode,
            "actions":       engineer_actions[failure_mode],
            "context_docs":  [r["doc"]["id"] for r in retrieved],
            "similar_cases": relevant_cases,
            "service_refs":  ["SP-001 Relay Replacement", "SP-002 Contact Cleaning"]
                             if failure_mode == "CONTACT_OXIDATION" else ["SP-001 Relay Replacement"],
            "raw_telemetry": alert,
        },

        # Image analysis (if provided)
        "visual_inspection": image_result,

        # RAG provenance
        "retrieval": {
            "query":          query,
            "sources_used":   [r["doc"]["id"] for r in retrieved],
            "source_scores":  {r["doc"]["id"]: round(r["score"], 4) for r in retrieved},
            "retriever_type": "TF-IDF (swap to your RAG embedder)",
        }
    }
    return report

## Section

In [ ]:
# 6. DEMO — Run the full pipeline end to end

## Section

In [ ]:
DEMO_ALERTS = [
    {
        "device_id":            "INV-JK-00142",
        "health_index":         22.5,
        "rul_days":             19.0,
        "rul_ci_low":           13.0,
        "rul_ci_high":          28.0,
        "arc_energy_sum_72h":   4.8,
        "temperature_mean_C":   38.0,
        "contact_resistance_mOhm": 185.0,
        "bounce_ms":            5.2,
        "is_anomaly":           False,
        "ac_starts_detected":   True,
        "is_coastal":           False,
        "scenario_note":        "Heavy AC user — arc erosion pattern",
    },
    {
        "device_id":            "INV-KL-00891",
        "health_index":         41.0,
        "rul_days":             55.0,
        "rul_ci_low":           44.0,
        "rul_ci_high":          70.0,
        "arc_energy_sum_72h":   0.3,
        "temperature_mean_C":   34.0,
        "contact_resistance_mOhm": 210.0,
        "bounce_ms":            2.8,
        "is_anomaly":           False,
        "ac_starts_detected":   False,
        "is_coastal":           True,
        "scenario_note":        "Coastal location — contact oxidation pattern",
    },
    {
        "device_id":            "INV-GJ-00033",
        "health_index":         8.0,
        "rul_days":             5.0,
        "rul_ci_low":           2.0,
        "rul_ci_high":          9.0,
        "arc_energy_sum_72h":   0.1,
        "temperature_mean_C":   52.0,
        "contact_resistance_mOhm": 95.0,
        "bounce_ms":            3.1,
        "is_anomaly":           True,
        "ac_starts_detected":   False,
        "is_coastal":           False,
        "scenario_note":        "Industrial pump house — thermal degradation",
    },
]


def run_demo(retriever: TFIDFRetriever):
    print("\n" + "=" * 60)
    print("DEMO — RAG Copilot Diagnostic Pipeline")
    print("=" * 60)

    reports = []
    for alert in DEMO_ALERTS:
        print(f"\n{'─'*55}")
        print(f"Device: {alert['device_id']}  |  {alert['scenario_note']}")
        print(f"{'─'*55}")

        # Simulate image path (in prod this comes from engineer's phone camera)
        visual_label = (
            "severe_erosion" if alert["health_index"] < 15 else
            "moderate_erosion" if alert["health_index"] < 40 else
            "mild_pitting"
        )
        img_path = f"photos/{alert['device_id']}_{visual_label}.jpg"

        report = generate_diagnostic_report(alert, retriever, image_path=img_path)
        reports.append(report)

        # Print key outputs
        a = report["alert"]
        print(f"\n  HI: {a['health_index']}%  |  Severity: {a['severity']}")
        print(f"  RUL: {a['rul_days']} days  [{report['alert']['rul_ci'][0]}–{report['alert']['rul_ci'][1]} days]")
        print(f"  Failure mode: {a['failure_mode']}")
        print(f"\n  Customer message:")
        print(f"    \"{report['customer']['message']}\"")
        print(f"\n  Engineer actions:")
        for step in report["engineer"]["actions"]:
            print(f"    • {step}")
        print(f"\n  Retrieved KB docs: {report['retrieval']['sources_used']}")
        print(f"  Visual inspection: {report['visual_inspection']['description']}")

        # Save report
        out_path = os.path.join(KB_DIR, f"report_{alert['device_id']}.json")
        with open(out_path, "w") as f:
            json.dump(report, f, indent=2)

    return reports

## Section

In [ ]:
# 7. MAIN

## Section

In [ ]:
def main():
    print("=" * 60)
    print("ARIA — Notebook 3: RAG Copilot Integration")
    print("=" * 60)

    docs      = build_knowledge_base()
    retriever = TFIDFRetriever(docs)
    print(f"  TF-IDF index built: {len(retriever.vocab)} vocab terms")
    print("  [SWAP_RAG] Replace TFIDFRetriever with your RAG embedder")

    # Demo: PDF ingestion stub
    print("\n  PDF ingestion stub (your RAG handles this in production):")
    pdf_result = ingest_pdf_datasheet("datasheets/omron_g2r.pdf")
    print(f"    {pdf_result['message']}")

    reports = run_demo(retriever)

    # Save KB index
    kb_index = {
        "documents":      [d["id"] for d in docs],
        "types":          list({d["type"] for d in docs}),
        "total":          len(docs),
        "retriever":      "TF-IDF (swap to your RAG embedder)",
        "swap_points":    {
            "PDF ingestion":     "ingest_pdf_datasheet() → your_rag.ingest_pdf()",
            "Image analysis":    "analyse_relay_image() → your_rag.analyse_image()",
            "Retrieval":         "TFIDFRetriever → YourRAGRetriever",
            "Report generation": "template-based → your_llm.generate(prompt, context)",
        },
        "built_at": datetime.now().isoformat(),
    }
    with open(os.path.join(KB_DIR, "kb_index.json"), "w") as f:
        json.dump(kb_index, f, indent=2)

    print(f"\n{'='*60}")
    print(f"KB documents: {len(docs)}")
    print(f"Demo reports saved to: ./{KB_DIR}/")
    print(f"Files: {sorted(os.listdir(KB_DIR))}")
    print("Next step → Notebook 4: FastAPI server + MQTT pipeline")
    print("=" * 60)


if __name__ == "__main__":
    main()

## Run Demo

In [ ]:
docs = build_knowledge_base()
retriever = TFIDFRetriever(docs)
print(f'KB: {len(docs)} docs | Vocab: {len(retriever.vocab)} terms')
print('[SWAP_RAG] Replace TFIDFRetriever with your RAG embedder')
reports = run_demo(retriever)
print(f'\n{len(reports)} diagnostic reports saved to aria_kb/')